# Taller recuperación de datos OSINT – SOCMINT
**Máster Universitario en Ciberseguridad · Ciberinteligencia · Taller OSINT**

Enunciado:
Se pretende que, usando las herramientas vistas en clase y extendiendo el notebook
proporcionado, se obtengan de la página getdaytrends los 50 trending topics del mes de
septiembre en horario de prime time (22:00) en España.
La entrega se realizará individualmente y contabilizará para el 10% de participación en las
actividades presenciales de la asignatura.

## 1 · Preparar el entorno

Instala Chrome en la máquina de Colab. Solo hay que hacerlo una vez por sesión.

# beautifulsoup4
Analiza el código HTML recibido para buscar, filtrar y extraer elementos específicos (como los nombres de las tendencias y el volumen de tweets).


In [1]:
# Instala Selenium y Google Chrome en la máquina de Colab (tarda 1-2 minutos)
!pip -q install selenium
!apt-get -qq update > /dev/null
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get -qq install -y ./google-chrome-stable_current_amd64.deb > /dev/null
!rm -f google-chrome-stable_current_amd64.deb
!google-chrome --version
!pip install requests beautifulsoup4 pandas


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Google Chrome 155.0.8059.39 


# 2 · Código para la extracción de datos
En el siguiente código se muestra el procedimiento empleado para la obtención de los datos. Se implementa un bucle iterativo que recorre de forma secuencial los días del periodo analizado, verificando e informando en tiempo real el estado de éxito de cada extracción.

In [2]:
import time, re
from datetime import date, timedelta
import pandas as pd
import requests
from bs4 import BeautifulSoup

def extraer_septiembre_22h_colab(year=2026, pais="spain"):
    todos_los_datos = []

    # Rango del 1 al 30 de septiembre
    inicio = date(year, 9, 1)
    fin = date(year, 9, 30)
    dias = [inicio + timedelta(days=i) for i in range((fin - inicio).days + 1)]

    # Encabezados para simular un navegador real desde Colab
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "es-ES,es;q=0.9",
        "Referer": "https://getdaytrends.com/"
    }

    print(f"--- Iniciando extracción rápida en Colab ({pais.capitalize()}, Septiembre {year}, 22:00h) ---")

    for fecha in dias:
        fecha_str = fecha.strftime("%Y-%m-%d")
        url = f"https://getdaytrends.com/es/{pais}/{fecha_str}/22/"

        try:
            res = requests.get(url, headers=headers, timeout=15)

            if res.status_code == 200:
                soup = BeautifulSoup(res.text, "html.parser")

                # Buscar el contenedor #trends
                tabla = soup.find(id="trends")
                if not tabla:
                    print(f"⚠️ No se encontró la tabla de tendencias para {fecha_str}")
                    continue

                enlaces = tabla.select('a[href*="/trend/"]')
                vistos = set()
                filas_dia = []

                for a in enlaces:
                    href = a.get("href", "")
                    if not href.startswith("http"):
                        href = "https://getdaytrends.com" + href

                    if href in vistos:
                        continue
                    vistos.add(href)

                    nombre = a.get_text(strip=True)
                    if not nombre:
                        continue

                    # Extraer volumen de tweets del td contenedor
                    volumen = None
                    td = a.find_parent("td")
                    if td:
                        m = re.search(r"(Menos de [\d.,]+K|[\d.,]+K?) tweets", td.get_text())
                        volumen = m.group(0) if m else None

                    filas_dia.append({
                        "fecha": fecha_str,
                        "hora": "22:00",
                        "posicion": len(filas_dia) + 1,
                        "tendencia": nombre,
                        "volumen": volumen,
                        "url_tendencia": href
                    })

                    if len(filas_dia) == 50:
                        break

                print(f"✓ {fecha_str}: Extraídas {len(filas_dia)} tendencias.")
                todos_los_datos.extend(filas_dia)
            else:
                print(f"✗ {fecha_str}: Servidor devolvió estado HTTP {res.status_code}")

        except Exception as e:
            print(f"⚠️ Error al conectar con el día {fecha_str}: {e}")

        time.sleep(1)  # Pausa breve entre peticiones

    df = pd.DataFrame(todos_los_datos)
    if not df.empty:
        df = df[["fecha", "hora", "posicion", "tendencia", "volumen", "url_tendencia"]]
    return df

# --- Ejecución ---
df_septiembre = extraer_septiembre_22h_colab(year=2026, pais="spain")

# Guardar DataFrame a CSV
if not df_septiembre.empty:
    df_septiembre.to_csv("tendencias_espana_septiembre_22h.csv", index=False, encoding="utf-8-sig")
    print(f"\n¡Éxito! Archivo guardado como 'tendencias_espana_septiembre_22h.csv' con {len(df_septiembre)} registros.")
else:
    print("\nNo se pudieron extraer datos.")



--- Iniciando extracción rápida en Colab (Spain, Septiembre 2026, 22:00h) ---
✓ 2026-09-01: Extraídas 50 tendencias.
✓ 2026-09-02: Extraídas 50 tendencias.
✓ 2026-09-03: Extraídas 50 tendencias.
✓ 2026-09-04: Extraídas 50 tendencias.
✓ 2026-09-05: Extraídas 50 tendencias.
✓ 2026-09-06: Extraídas 50 tendencias.
✓ 2026-09-07: Extraídas 50 tendencias.
✓ 2026-09-08: Extraídas 50 tendencias.
✓ 2026-09-09: Extraídas 50 tendencias.
✓ 2026-09-10: Extraídas 50 tendencias.
✓ 2026-09-11: Extraídas 47 tendencias.
✓ 2026-09-12: Extraídas 50 tendencias.
✓ 2026-09-13: Extraídas 50 tendencias.
✓ 2026-09-14: Extraídas 50 tendencias.
✓ 2026-09-15: Extraídas 50 tendencias.
✓ 2026-09-16: Extraídas 50 tendencias.
✓ 2026-09-17: Extraídas 50 tendencias.
✓ 2026-09-18: Extraídas 50 tendencias.
✓ 2026-09-19: Extraídas 50 tendencias.
✓ 2026-09-20: Extraídas 50 tendencias.
✓ 2026-09-21: Extraídas 50 tendencias.
✓ 2026-09-22: Extraídas 47 tendencias.
✓ 2026-09-23: Extraídas 50 tendencias.
✓ 2026-09-24: Extraídas 4

## 6 · Extraer las tendencias y pasarlas a un DataFrame



Este bloque de código se encarga del procesamiento posterior, la trazabilidad y la presentación final de los datos recolectados.

**Sus funciones principales son:**
Lectura y Consolidación: Carga el conjunto de datos recién extraído (tendencias_espana_septiembre_22h.csv) en una estructura tabular de pandas.

**Generación de Metadatos y Auditoría (OSINT)**:
pais: Asigna la constante geográfica (spain) para garantizar la correcta categorización de la muestra.

**capturado_utc:** Genera una marca temporal en estándar internacional ISO-8601 en tiempo universal coordinado ($UTC$). Esto asegura la validez cronológica de la evidencia obtenida en las tareas de ciberinteligencia.

**Persistencia de Datos:** Sobreescribe el archivo CSV actualizando la estructura final con todas las variables de contexto e hipervínculos de origen.

**Validación Visual:** Muestra un resumen del total de registros procesados y despliega una vista previa de las primeras 15 tendencias ordenadas por posición para su inspección inmediata dentro del cuaderno.

In [3]:
from datetime import datetime, timezone
import pandas as pd

# 1. Cargar el archivo generado
df = pd.read_csv("tendencias_espana_septiembre_22h.csv")

# 2. Añadir las columnas de metadatos
PAIS = "spain"
df["pais"] = PAIS
df["capturado_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")

# 3. Guardar el CSV actualizado con los nuevos campos
df.to_csv("tendencias_espana_septiembre_22h.csv", index=False, encoding="utf-8-sig")

# 4. Imprimir resumen y visualizar las primeras 15 filas
print(f"✓ {len(df)} tendencias cargadas")
df.head(15)


✓ 1479 tendencias cargadas


,fecha,hora,posicion,tendencia,volumen,url_tendencia,pais,capturado_utc
0,2026-09-01,22:00,1,#codigo10,NaN,https://getdaytrends.com/es/spain/trend/%23cod...,spain,2026-10-08T14:00:39+00:00
1,2026-09-01,22:00,2,Pablo García,NaN,https://getdaytrends.com/es/spain/trend/Pablo%...,spain,2026-10-08T14:00:39+00:00
2,2026-09-01,22:00,3,Depor,NaN,https://getdaytrends.com/es/spain/trend/Depor/,spain,2026-10-08T14:00:39+00:00
3,2026-09-01,22:00,4,Losada,NaN,https://getdaytrends.com/es/spain/trend/Losada/,spain,2026-10-08T14:00:39+00:00
4,2026-09-01,22:00,5,#DogHouse,NaN,https://getdaytrends.com/es/spain/trend/%23Dog...,spain,2026-10-08T14:00:39+00:00
5,2026-09-01,22:00,6,Fajardo,NaN,https://getdaytrends.com/es/spain/trend/Fajardo/,spain,2026-10-08T14:00:39+00:00
6,2026-09-01,22:00,7,Thomas Lemar,NaN,https://getdaytrends.com/es/spain/trend/Thomas...,spain,2026-10-08T14:00:39+00:00
7,2026-09-01,22:00,8,Tagliafico,NaN,https://getdaytrends.com/es/spain/trend/Taglia...,spain,2026-10-08T14:00:39+00:00
8,2026-09-01,22:00,9,Cibeles,NaN,https://getdaytrends.com/es/spain/trend/Cibeles/,spain,2026-10-08T14:00:39+00:00
9,2026-09-01,22:00,10,Ceballos,NaN,https://getdaytrends.com/es/spain/trend/Ceballos/,spain,2026-10-08T14:00:39+00:00


# 4.Guardar la evidencia
Mañana estas tendencias habrán cambiado. Guardamos la URL, la fecha y hora, una captura, la copia de la página y el contexto, con el hash SHA-256 de cada fichero para poder demostrar que no se han modificado.

In [4]:
import os, json, hashlib
from datetime import datetime, timezone
import requests
import pandas as pd

# --- Configuración de Evidencias OSINT ---
PAIS = "spain"
# Guardamos la URL de muestra histórica representativa del estudio
URL_FUENTE_MUESTRA = f"https://getdaytrends.com/es/{PAIS}/2026-09-01/22/"
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36"
DIR_EVIDENCIAS = "evidencias"

os.makedirs(DIR_EVIDENCIAS, exist_ok=True)

# Marca de tiempo de la auditoría / generación de evidencia
timestamp_utc = datetime.now(timezone.utc)
str_timestamp = timestamp_utc.strftime("%Y%m%dT%H%M%SZ")
str_iso_utc = timestamp_utc.isoformat(timespec="seconds")

# Rutas estandarizadas de evidencia
ruta_html = os.path.join(DIR_EVIDENCIAS, f"getdaytrends_{str_timestamp}.html")
ruta_csv = os.path.join(DIR_EVIDENCIAS, f"getdaytrends_{str_timestamp}.csv")
ruta_json = os.path.join(DIR_EVIDENCIAS, f"getdaytrends_{str_timestamp}.json")

# 1. Descargar el HTML fuente de muestra del dataset histórico
headers = {"User-Agent": USER_AGENT, "Accept-Language": "es-ES,es;q=0.9"}
res_html = requests.get(URL_FUENTE_MUESTRA, headers=headers, timeout=15)
with open(ruta_html, "w", encoding="utf-8") as f:
    f.write(res_html.text)

# 2. Cargar CSV y añadir metadatos de auditoría sin alterar la fecha/hora original de los datos
df = pd.read_csv("tendencias_espana_septiembre_22h.csv")
df["pais"] = PAIS
df["auditoria_utc"] = str_iso_utc
df.to_csv(ruta_csv, index=False, encoding="utf-8-sig")

# 3. Función auxiliar para el Hash SHA-256
def calcular_sha256(ruta_archivo):
    sha256_hash = hashlib.sha256()
    with open(ruta_archivo, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

# 4. Generación del Manifiesto JSON de Cadena de Custodia
manifiesto_evidencia = {
    "url_muestra": URL_FUENTE_MUESTRA,
    "capturado_utc": str_timestamp,
    "periodo_analizado": "2026-09-01 al 2026-09-30 (22:00h UTC)",
    "user_agent": USER_AGENT,
    "herramienta": "Python Requests & BeautifulSoup4",
    "filas": len(df),
    "ficheros": {
        "html_muestra": {
            "ruta": ruta_html,
            "sha256": calcular_sha256(ruta_html)
        },
        "csv_completo": {
            "ruta": ruta_csv,
            "sha256": calcular_sha256(ruta_csv)
        }
    }
}

# Guardar manifiesto
with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump(manifiesto_evidencia, f, indent=2, ensure_ascii=False)

print("✓ Evidencias OSINT generadas con éxito:\n")
print(json.dumps(manifiesto_evidencia, indent=2, ensure_ascii=False))


✓ Evidencias OSINT generadas con éxito:

{
  "url_muestra": "https://getdaytrends.com/es/spain/2026-09-01/22/",
  "capturado_utc": "20261008T140051Z",
  "periodo_analizado": "2026-09-01 al 2026-09-30 (22:00h UTC)",
  "user_agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36",
  "herramienta": "Python Requests & BeautifulSoup4",
  "filas": 1479,
  "ficheros": {
    "html_muestra": {
      "ruta": "evidencias/getdaytrends_20261008T140051Z.html",
      "sha256": "e487b3111a5bdf72a178672336aed2fbe55c1687f1f1b1b3f6195bbb6fc37d89"
    },
    "csv_completo": {
      "ruta": "evidencias/getdaytrends_20261008T140051Z.csv",
      "sha256": "ba5a635316c219c48b3b277af998a1675b574c7b97ed598a68c62ea617e60dfa"
    }
  }
}


# 5. Cierre del proceso

Como el proceso actual en Colab utiliza **requests** y **BeautifulSoup** (peticiones HTTP directas sin abrir ventana de navegador), no hay ninguna instancia de Chrome abierta que requiera cierre.

Sin embargo, para cumplir formalmente con la metodología de recolección en tu reporte y dejar asentado el cierre explícito de la sesión de trabajo y liberación de recursos en memoria, puedes incluir este bloque final de cierre:

¿Por qué incluirlo en el reporte?
Gestión de Recursos: Demuestra buenas prácticas en la gestión de memoria RAM dentro de entornos virtuales como Colab.

Cierre de Auditoría: Marca formalmente el fin de la ventana de captura en la cadena de custodia OSINT.

In [5]:
import gc

# 1. Liberar memoria RAM eliminando los DataFrames y objetos pesados de la sesión
try:
    del df
    del df_septiembre
    del todos_los_datos
except NameError:
    pass

# 2. Forzar la recolección de basura del sistema
gc.collect()

# 3. Notificación final de cierre de proceso para el reporte OSINT
print("==========================================================")
print("  PROCESO DE RECOLECCIÓN OSINT/SOCMINT FINALIZADO")
print("  - Sesión de red finalizada.")
print("  - Recolector de basura ejecutado (Memoria liberada).")
print("  - Evidencias y manifiesto JSON preservados en '/evidencias'.")
print("==========================================================")


  PROCESO DE RECOLECCIÓN OSINT/SOCMINT FINALIZADO
  - Sesión de red finalizada.
  - Recolector de basura ejecutado (Memoria liberada).
  - Evidencias y manifiesto JSON preservados en '/evidencias'.
